# Evaluation Comparison

Benchmark all trained models using BLEU, ROUGE, and Exact Match.

In [ ]:
import os, sys, logging, warnings

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]      = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"]            = "error"
os.environ["TOKENIZERS_PARALLELISM"]            = "false"
os.environ["TRL_DISABLE_RICH"]                  = "1"
os.environ["TORCHAO_DISABLE_CPP_EXTENSIONS"]    = "1"
os.environ["TORCHAO_SKIP_CPP_LOAD"]             = "1"
os.environ["USE_TORCHAO"]                       = "0"
os.environ['HF_TOKEN'] = 'hf_...'   # your huggingface token

In [ ]:
import importlib, sys
import gc, traceback
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset

from src.utils.config_loader import load_config
from src.data.preprocess import (
    load_stackoverflow, split_qa, make_eval_frame,
)
from src.eval.generate import generate_predictions
from src.eval.metrics  import compute_all_metrics
from src.models.loaders import load_model_for_inference, load_tokenizer

### Load Test Set

In [ ]:
eval_cfg = load_config("configs/eval.yaml")

DATA_DIR = "/a-survey-of-LLMs-fine-tuning-approaches/data/stackoverflow/stacksample"

df = load_stackoverflow(DATA_DIR)
_, _, test_df = split_qa(df)
eval_df = make_eval_frame(test_df)
print(f"Test samples: {len(eval_df)}")
print("Columns:", list(eval_df.columns))

### Define evaluate_model Helper

In [ ]:
NUM_SAMPLES     = eval_cfg.get("num_samples", 20)
MAX_NEW_TOKENS  = eval_cfg.get("max_new_tokens", 128)
MAX_INPUT_LEN   = eval_cfg.get("max_input_length", 512)
EVAL_BATCH_SIZE = eval_cfg.get("batch_size", 4)


def evaluate_model(adapter_or_model_path, num_samples=NUM_SAMPLES):
    """Load a full model OR a PEFT adapter and return (metrics, preds, refs)."""
    model, tokenizer_source = load_model_for_inference(
        adapter_or_model_path, dtype="auto", device_map="auto",
    )
    tokenizer = load_tokenizer(tokenizer_source, padding_side="left")

    subset  = eval_df.head(num_samples)
    prompts = subset["prompt"].tolist()
    refs    = subset["reference"].tolist()

    preds = generate_predictions(
        model, tokenizer, prompts,
        max_new_tokens=MAX_NEW_TOKENS,
        batch_size=EVAL_BATCH_SIZE,
        max_input_length=MAX_INPUT_LEN,
    )

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return compute_all_metrics(preds, refs), preds, refs

### Benchmark All Models

In [ ]:
MODELS = eval_cfg["models"]
print("Models to evaluate:")
for k, v in MODELS.items():
    print(f"  {k:10s} -> {v}  (exists={os.path.isdir(v)})")

rows = []
sample_dumps = {}
for name, path in MODELS.items():
    if not os.path.isdir(path):
        print(f"[SKIP] {name}: path does not exist -> {path}")
        continue
    try:
        metrics, preds, refs = evaluate_model(path)
        metrics["method"] = name
        rows.append(metrics)
        sample_dumps[name] = list(zip(preds, refs))
        print(f"[OK] {name}: bleu={metrics['bleu']:.4f}  "
              f"rougeL={metrics['rougeL']:.4f}  "
              f"EM={metrics['exact_match']:.4f}  n={metrics['n']}")
    except Exception as e:
        print(f"[FAIL] {name}: {type(e).__name__}: {e}")
        traceback.print_exc()

results = pd.DataFrame(rows)
if not results.empty:
    results = results[["method", "bleu", "rougeL", "exact_match", "n"]]
results

### Plot Comparison

In [ ]:
PLOT_METRICS = ["bleu", "rougeL", "exact_match"]
REPO = "/a-survey-of-LLMs-fine-tuning-approaches"

if results.empty:
    print("No results to plot.")
else:
    plot_df = results.set_index("method")[PLOT_METRICS]
    ax = plot_df.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Fine-Tuning Method Comparison")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=0)
    plt.tight_layout()

    out_png = eval_cfg.get("output_png",
                           f"{REPO}/outputs/evaluation_comparison.png")
    if not os.path.isabs(out_png):
        out_png = f"{REPO}/{out_png}"
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    plt.savefig(out_png, dpi=150)
    plt.show()
    print("Saved:", out_png)

### Inspect a few predictions

In [ ]:
if sample_dumps:
    name = list(sample_dumps.keys())[0]
    print(f"=== {name} — 3 samples ===")
    for i, (pred, ref) in enumerate(sample_dumps[name][:3]):
        print(f"\n--- sample {i} ---")
        print("REF :", ref[:300].replace("\n", " "))
        print("PRED:", pred[:300].replace("\n", " "))